In [ ]:
import cvxpy as cp
import numpy as np
import pandas as pd
from enum import Enum

class Season(Enum):
	WINTER = ('jan', 91)
	SPRING = ('apr', 91)
	SUMMER = ('jul', 91)
	AUTUMN = ('oct', 92)

I = 1
DELTA_t = 1.0
HOURS_PER_DAY = 24
RETIREMENT_SOH = 0.80  # Retirement at 20% capacity loss


SOC_MIN, SOC_MAX = 0.05, 0.95
Q_NEW = 3.916
P_MAX = 0.979
T_MIN, T_REF, T_MAX = -30, 25, 50
ETA = 0.968

ALPHA = 2.61e-5
BETA, GAMMA = 2.5, 3.0
KAPPA = 0.04
RHO, SIGMA = 19500, 241000
A, B = 0.02, 0.02

expected_prices_df = pd.read_csv("./data/raw/DAM/seasonal_stats.csv")

def solve_bess_block(aging, soh_init, soc_init, temp_init, season_enum):
	code, days = season_enum.value
	n_steps = days * HOURS_PER_DAY

	lambda_daily = expected_prices_df[f'{code}_mean'].to_numpy().reshape(-1, 1)
	prices = np.tile(lambda_daily, (days, 1))

	E = cp.Variable((n_steps + 1, I), nonneg=True)
	Q = cp.Variable((n_steps + 1, I), nonneg=True)
	T = cp.Variable((n_steps + 1, I), nonneg=True)
	c = cp.Variable((n_steps, I), nonneg=True)
	d = cp.Variable((n_steps, I), nonneg=True)
	u = cp.Variable((n_steps, I), nonneg=True)

	phi = cp.Parameter((n_steps, I), nonneg=True)
	delta_Q = ALPHA * cp.multiply(phi, c + d) * DELTA_t

	constraints = [
		Q[0, :] == Q_NEW * soh_init,
		E[0, :] == Q[0, :] * soc_init,
		T[0, :] == temp_init,
		Q[1:, :] == Q[:-1, :] - delta_Q,
		E[1:, :] == E[:-1, :] + (ETA * c * DELTA_t) - (d * DELTA_t / ETA),
		T[1:, :] == T[:-1, :] + (BETA * (c + d) * DELTA_t) - (GAMMA * u * DELTA_t),
		E >= Q * SOC_MIN, E <= Q * SOC_MAX,
		T >= T_MIN, T <= T_MAX,
		c <= P_MAX, d <= P_MAX, u <= 1,
		E[n_steps, :] >= E[0, :] # Terminal SOC balance
	]

	arb_cost = cp.sum(cp.multiply(prices, c - d) * DELTA_t)
	oper_cost = cp.sum(cp.multiply(prices, A + B * u) * P_MAX * DELTA_t)
	opport_cost = cp.sum(RHO * delta_Q)
	replac_cost = cp.sum(SIGMA * delta_Q)
	term_cost = cp.sum((A + B) * prices.mean() * cp.pos(T[-1, :] - T_REF) / (GAMMA * DELTA_t))

	if aging:
		obj = cp.Minimize(arb_cost + oper_cost + opport_cost + replac_cost + term_cost)
	else:
		obj = cp.Minimize(arb_cost + oper_cost + term_cost)

	prob = cp.Problem(obj, constraints)
	t_est = np.full((n_steps, I), T_REF)

	for _ in range(5):
		phi.value = 1 + KAPPA * np.maximum(0, t_est - T_REF)
		prob.solve(solver=cp.ECOS, feastol=1e-8)
		if prob.status != "optimal": break
		t_est = T.value[:-1, :]

	res = {
		'rev': cp.sum(cp.multiply(prices, d) * DELTA_t).value,
		'charge': cp.sum(cp.multiply(prices, c) * DELTA_t).value,
		'oper': oper_cost.value,
		'opport': opport_cost.value,
		'replac': replac_cost.value,
		'soh_end': Q.value[-1, 0] / Q_NEW,
		'soc_end': E.value[-1, 0] / Q.value[-1, 0],
		'temp_end': T.value[-1, 0]
	}
	return res

def run_lifecycle(is_aging_aware):
	tally = {'rev': 0, 'charge': 0, 'oper': 0, 'opport': 0, 'replac': 0, 'days': 0}
	curr_soh = 1.0
	curr_soc = 0.5
	curr_temp = T_REF

	while curr_soh > RETIREMENT_SOH:
		for s in Season:
			result = solve_bess_block(is_aging_aware, curr_soh, curr_soc, curr_temp, s)

			for key in ['rev', 'charge', 'oper', 'opport', 'replac']:
				tally[key] += result[key]
			tally['days'] += s.value[1]

			curr_soh = result['soh_end']
			curr_soc = result['soc_end']

			if curr_soh <= RETIREMENT_SOH:
				break

	tally['net_profit'] = tally['rev'] - (tally['charge'] + tally['oper'] + tally['opport'] + tally['replac'])
	return tally

aware_life = run_lifecycle(True)
agnostic_life = run_lifecycle(False)

In [ ]:
print(f"{'Metric':<35} | {'Aging-Aware':<15} | {'Aging-Agnostic':<15}")
print("-" * 72)
print(f"{'Lifetime (Days)':<35} | {aware_life['days']:<15} | {agnostic_life['days']:<15}")
print(f"{'Lifetime (Years)':<35} | {aware_life['days']/365:<15.2f} | {agnostic_life['days']/365:<15.2f}")
print(f"{'Revenue ($)':<35} | {aware_life['rev']:<15.2f} | {agnostic_life['rev']:<15.2f}")
print(f"{'Charging Cost ($)':<35} | {aware_life['charge']:<15.2f} | {agnostic_life['charge']:<15.2f}")
print(f"{'Operational Cost ($)':<35} | {aware_life['oper']:<15.2f} | {agnostic_life['oper']:<15.2f}")
print(f"{'Opportunity Cost ($)':<35} | {aware_life['opport']:<15.2f} | {agnostic_life['opport']:<15.2f}")
print(f"{'Replacement Cost ($)':<35} | {aware_life['replac']:<15.2f} | {agnostic_life['replac']:<15.2f}")
print(f"{'Total Net Profit ($)':<35} | {aware_life['net_profit']:<15.2f} | {agnostic_life['net_profit']:<15.2f}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.ticker import FuncFormatter

data = {
	'Metric': ['Aging\nAware', 'Aging\nAgnostic'],
	'Charging Cost': [314713.99, 402298.42],
	'Operational Cost': [103736.40, 69710.85],
	'Opportunity Cost': [15532.17, 15380.23],
	'Replacement Cost': [191961.74, 190083.87],
	'Total Net Profit': [350554.20, 168490.35]
}

df = pd.DataFrame(data)

df['Total Revenue'] = df.iloc[:, 1:].sum(axis=1)
df = df.sort_values(by='Total Revenue', ascending=True)

categories = ['Charging Cost', 'Operational Cost', 'Opportunity Cost', 'Replacement Cost', 'Total Net Profit']
colors = ['#d62728', '#1f77b4', '#ff7f0e', '#9467bd', '#2ca02c'] # Red, Blue, Orange, Purple, Green

plt.figure(figsize=(12, 5))

left_sum = np.zeros(len(df))

# Plot horizontal bars
for i, category in enumerate(categories):
	plt.barh(df['Metric'], df[category], left=left_sum, label=category, color=colors[i])
	left_sum += df[category].values

def thousands_formatter(x, pos):
	return f'{x/1000:,.0f}k'

plt.gca().xaxis.set_major_formatter(FuncFormatter(thousands_formatter))

plt.title('Lifetime Revenue Breakdown: Aging-Aware vs. Aging-Agnostic')
plt.xlabel('Revenue ($ Thousands)')
plt.ylabel('')
plt.xlim(0, 1200000)

plt.legend(loc='lower right', frameon=True, shadow=True)

plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.tight_layout()

plt.savefig('img/revenue_comparison.png')